
1. Tek bir self-attention head kurmak: **Query (Q)**, **Key (K)**, **Value (V)** ve **Pozisyon Embedding'i** eklemek.
2. Dikkat ağırlık matrisini (`wei`) hesaplayıp yazdırmak, seçilen bir satırını detaylı okuyarak o token'ın hangi geçmiş token'lara ne oranda baktığını analiz etmek.
3. Dikkat skorlarını neden $\sqrt{\text{head\_size}}$ değerine böldüğümüzü sayısal veriler ve varyans hesaplarıyla göstermek: bölmeden ve bölerek elde edilen Softmax çıktılarını yan yana karşılaştırmak.


In [3]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import pandas as pd


In [ ]:
torch.manual_seed(42)
B, T, C = 4, 8, 32 # Batch=4, Context=8, Embedding=32
head_size = 16
vocab_size = 65

# Token ve pozisyon embedding tabloları
token_embedding_table = nn.Embedding(vocab_size, C)
position_embedding_table = nn.Embedding(T, C)

# Rastgele  girdi dizisi (B, T)
idx = torch.randint(0, vocab_size, (B, T))
tok_emb = token_embedding_table(idx) # (B, T, C)
pos_emb = position_embedding_table(torch.arange(T)) # (T, C)

# Token içeriği + Pozisyon bilgisi
x = tok_emb + pos_emb # (B, T, C)
print("x.shape (B, T, C):", x.shape)

x.shape (B, T, C): torch.Size([4, 8, 32])


In [ ]:
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x)   # (B, T, head_size) -> (4,8,16)
q = query(x) # (B, T, head_size)
v = value(x) # (B, T, head_size)

# Attention skoru
wei = q @ k.transpose(-2, -1) * (head_size ** -0.5) # (B, T, T)
# Sayılar çok büyüdüğünde softmax hatalı sonuçlar çıkarmaya başlar bu yüzden 1/karekök alınarak küçültülüyor

# Causal Mask
tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1) # (B, T, T)

# Value vektörleriyle ağırlıklı toplam
out = wei @ v # (B, T, head_size)

print("wei.shape", wei.shape)
print("out.shape", out.shape)

wei.shape torch.Size([4, 8, 8])
out.shape torch.Size([4, 8, 16])


### wei[0] Ağırlık matrisi

In [ ]:
torch.set_printoptions(precision=4, sci_mode=False)
print(wei[0])

İlk batch elemanının dikkat ağırlık matrisi wei[0]:
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.8501, 0.1499, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4874, 0.2415, 0.2711, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0676, 0.4771, 0.3919, 0.0634, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4606, 0.1537, 0.0531, 0.0841, 0.2485, 0.0000, 0.0000, 0.0000],
        [0.0788, 0.2011, 0.1509, 0.1415, 0.1775, 0.2502, 0.0000, 0.0000],
        [0.2132, 0.0767, 0.2724, 0.0739, 0.1793, 0.0974, 0.0871, 0.0000],
        [0.0403, 0.0509, 0.0579, 0.1008, 0.0320, 0.2232, 0.4477, 0.0472]],
       grad_fn=<SelectBackward0>)


### Hangi tokene ne kadar bakıyor

In [7]:
# 4. indeksli token'ı (5. karakter) inceleyelim
target_idx = 4
row_weights = wei[0, target_idx]

print(f"=== {target_idx}. İndeksteki Token'ın Dikkat Dağılımı ===")
for j in range(T):
    val = row_weights[j].item()
    if j > target_idx:
        durum = "GELECEK -> Maskelendi (-inf sonrası Softmax = 0.00%)"
    elif j == target_idx:
        durum = "KENDİSİ -> Mevcut token konumu"
    else:
        durum = "GEÇMİŞ  -> Önceki bağlam token'ı"
    print(f"Hedef Token {target_idx} ---> Kaynak Token {j}: {val:.4f} (%{val*100:6.2f}) | {durum}")

print(f"\nSatır Toplamı (Doğrulama): {row_weights.sum().item():.4f}")

=== 4. İndeksteki Token'ın Dikkat Dağılımı ===
Hedef Token 4 ---> Kaynak Token 0: 0.4606 (% 46.06) | GEÇMİŞ  -> Önceki bağlam token'ı
Hedef Token 4 ---> Kaynak Token 1: 0.1537 (% 15.37) | GEÇMİŞ  -> Önceki bağlam token'ı
Hedef Token 4 ---> Kaynak Token 2: 0.0531 (%  5.31) | GEÇMİŞ  -> Önceki bağlam token'ı
Hedef Token 4 ---> Kaynak Token 3: 0.0841 (%  8.41) | GEÇMİŞ  -> Önceki bağlam token'ı
Hedef Token 4 ---> Kaynak Token 4: 0.2485 (% 24.85) | KENDİSİ -> Mevcut token konumu
Hedef Token 4 ---> Kaynak Token 5: 0.0000 (%  0.00) | GELECEK -> Maskelendi (-inf sonrası Softmax = 0.00%)
Hedef Token 4 ---> Kaynak Token 6: 0.0000 (%  0.00) | GELECEK -> Maskelendi (-inf sonrası Softmax = 0.00%)
Hedef Token 4 ---> Kaynak Token 7: 0.0000 (%  0.00) | GELECEK -> Maskelendi (-inf sonrası Softmax = 0.00%)

Satır Toplamı (Doğrulama): 1.0000


### neden karekök(headsize) değerine bölüyoruz

In [ ]:
torch.manual_seed(42)
k_test = torch.randn(B, T, head_size)
q_test = torch.randn(B, T, head_size)

wei_unscaled = q_test @ k_test.transpose(-2, -1)

wei_scaled = q_test @ k_test.transpose(-2, -1) * (head_size ** -0.5)

print(f"k_test varyansı                 : {k_test.var().item():.4f}")
print(f"q_test varyansı                 : {q_test.var().item():.4f}")
print(f"Ölçeklenmemiş wei varyansı      : {wei_unscaled.var().item():.4f}  (yaklaşık head_size = {head_size})")
print(f"Ölçeklenmiş wei varyansı (x 1/√h): {wei_scaled.var().item():.4f}    (varyans tekrar ~1.0 seviyesinde)")

k_test varyansı                 : 1.0877
q_test varyansı                 : 1.0594
Ölçeklenmemiş wei varyansı      : 18.4436  (yaklaşık head_size = 16)
Ölçeklenmiş wei varyansı (x 1/√h): 1.1527    (varyans tekrar ~1.0 seviyesinde)


In [ ]:
# scale and unscale
raw_wei = q @ k.transpose(-2, -1)
raw_wei_masked = raw_wei.masked_fill(tril == 0, float('-inf'))

wei_scaled_sm = F.softmax(raw_wei_masked * (head_size ** -0.5), dim=-1)[0, -1]
wei_unscaled_sm = F.softmax(raw_wei_masked, dim=-1)[0, -1]

df_mat = pd.DataFrame({
    "Token İndeksi": list(range(T)),
    "Softmax (Ölçeklenmiş - 1/√head_size)": [f"{v.item():.4f} (%{v.item()*100:5.2f})" for v in wei_scaled_sm],
    "Softmax (Bölünmemiş / Ölceksiz)": [f"{v.item():.4f} (%{v.item()*100:5.2f})" for v in wei_unscaled_sm],
})
print("\nGerçek Veri Üzerinde Son Token'ın Dikkat Dağılımı:")
print(df_mat.to_string(index=False))


Gerçek Veri Üzerinde Son Token'ın Dikkat Dağılımı:
 Token İndeksi Softmax (Ölçeklenmiş - 1/√head_size) Softmax (Bölünmemiş / Ölceksiz)
             0                      0.0403 (% 4.03)                 0.0001 (% 0.01)
             1                      0.0509 (% 5.09)                 0.0002 (% 0.02)
             2                      0.0579 (% 5.79)                 0.0003 (% 0.03)
             3                      0.1008 (%10.08)                 0.0024 (% 0.24)
             4                      0.0320 (% 3.20)                 0.0000 (% 0.00)
             5                      0.2232 (%22.32)                 0.0580 (% 5.80)
             6                      0.4477 (%44.77)                 0.9390 (%93.90)
             7                      0.0472 (% 4.72)                 0.0001 (% 0.01)
